In [ ]:
import os
from faker import Faker
import json
import uuid
import time

fake = Faker()
# Matches the DEFAULT_LANDING_PATH in your SDP-ingest_prod.py
LANDING_ZONE = "/Volumes/lakehouse_dev/cloudwatch_metrics_pull/raw_data/prod/users_json"

def generate_user_event():
    return {
        "id": fake.random_int(min=1, max=1000), # Using IDs 1-1000 to simulate upserts
        "email": fake.email(),
        "creation_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"),
        "last_activity_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"),
        "firstname": fake.first_name(),
        "lastname": fake.last_name(),
        "city": fake.city(),
        "postcode": fake.postcode()
    }

def run_simulation(batch_size=5):
    events = [generate_user_event() for _ in range(batch_size)]

    # Corrected: Use .hex and slice it for a 'short' version
    unique_id = uuid.uuid4().hex[:8]
    file_name = f"sim_batch_{int(time.time())}_{unique_id}.json"

    # Convert list of dicts to newline-delimited JSON
    json_data = "\n".join([json.dumps(e) for e in events])

    # Use dbutils for better compatibility with Databricks Volumes
    dbutils.fs.put(f"{LANDING_ZONE}/{file_name}", json_data, overwrite=True)

    print(f"Uploaded {batch_size} events to {LANDING_ZONE}")

run_simulation(10)